# 33 — Extracción con modelos de pago: seis modelos frente a `qwen3:8b`

**Autor:** Jason Sepúlveda S.
**Proyecto:** Evaluación experimental de la extracción automática y la representación
persistente de decisiones organizacionales frente a Recuperación Aumentada (RAG)
**Depende de:** [ADR 0026](../docs/decisions/0026-adr-extension-extractor-con-modelos-hosted.md) ·
[ADR 0011](../docs/decisions/0011-adr-alineacion-rol-e-intervalos-del-experimento-a.md) ·
[ADR 0012](../docs/decisions/0012-adr-umbral-tau-del-experimento-a.md) ·
[notebook 28](28-jss-experimento-a.ipynb) ·
[protocolo de la extensión](../docs/protocols/extraccion-hosted.md)

**Alcance.** Registro de análisis de la extensión descriptiva del Experimento A: la extracción de
decisiones de C2 se repite con seis modelos de pago (dos por laboratorio) y se compara con la
corrida formal `experimento-a-formal`. El notebook lee las corridas guardadas en
`reports/experiment-a/` y los espacios de trabajo de OpenKOS en `data/interim/openkos/`; **no llama
a ningún modelo generativo**. Toda cifra sale de funciones de `src/afg/experiments/` con test
(`hosted_comparison.py` y `hosted_notebook.py`); el notebook solo las invoca y las muestra.

**Estado.** Descriptivo (ADR 0026, decisión 9): las series de evaluación ya se usaron en la corrida
formal, de modo que ningún resultado de este notebook confirma una hipótesis de la tesis. El
notebook se vuelve a ejecutar después de cada corrida; las secciones sin datos muestran el aviso
**pendiente** en vez de fallar. **Requisito:** cuando existen corridas, el cálculo de similitudes
necesita Ollama en marcha con el modelo de embeddings `bge-m3`.

> **Aviso de alcance (ADR 0005).** Las series son series de **evaluación**. El notebook muestra solo
> **identificadores de serie, nombres de modelo y cifras agregadas**. No muestra ni cita frases de
> decisión ni contenido de transcripciones. No contiene ninguna cifra del Experimento B.

## Los términos, en simple

| Término | Qué significa |
|---|---|
| Extracción | El paso en que OpenKOS lee las transcripciones de las reuniones de una serie y escribe las decisiones que encuentra. |
| Línea base | La corrida formal `experimento-a-formal`, hecha con el modelo local `qwen3:8b`. No se vuelve a correr. |
| Referencia | Las decisiones que anotó el equipo a mano en cada serie; es contra lo que se mide la extracción. |
| τ (tau) | Umbral de similitud: una decisión extraída cuenta como la de la referencia cuando su similitud con ella supera τ. El principal es 0,70; 0,75 y 0,80 son sensibilidades. |
| Precision, recall, F1 | De lo extraído, qué parte está en la referencia (precision); de la referencia, qué parte se encontró (recall); y su media armónica (F1). |
| Bootstrap pareado | Se vuelve a calcular la diferencia entre un modelo y la línea base 10.000 veces, tomando en cada vuelta las mismas reuniones al azar para los dos sistemas. El rango que cubre el 95 % central de esas diferencias es el intervalo. |
| Bonferroni | Corrección por probar seis modelos a la vez: cada intervalo se calcula con el nivel 1 − 0,05/6, más exigente que 95 %, para que el error total no pase de 5 %. |
| `ingest` | El comando de OpenKOS que procesa una reunión; su duración es el tiempo de extracción de esa reunión. |
| `reasoning_effort` | Parámetro que indica cuánto «razona» un modelo antes de responder. Se fija en `low` para los seis. |
| Puente de continuidad | Corrida de comprobación: el mismo `qwen3:8b` con la versión nueva de OpenKOS (0.7.2), para saber si el cambio de versión por sí solo altera las cifras. |

---

# 1. Pregunta, diseño y regla prerregistrada

**Pregunta.** La extracción automática de decisiones es el eslabón débil de C2: con `qwen3:8b`
encuentra menos de un tercio de las decisiones de la referencia. ¿Un modelo de pago más capaz
extrae más decisiones, sin perder precision?

**Qué se hace.** Se leen la configuración (`config/hosted_extraction.toml`) y se muestran los seis
modelos, los parámetros compartidos y la regla de decisión que el ADR 0026 fijó antes de ver datos.

In [1]:
import json
import warnings

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import Markdown, display

from afg.cli import make_alignment_similarity
from afg.evaluation.reference import load_series_reference
from afg.experiments.hosted_comparison import (
    build_comparison,
    ingest_time_summary,
    load_run_data,
)
from afg.experiments.hosted_extraction import load_hosted_config
from afg.experiments.hosted_notebook import (
    baseline_figures,
    bridge_frame,
    bridge_state,
    format_es,
    format_interval,
    models_frame,
    operation_table,
    quality_table,
    recall_forest_rows,
    run_status,
    time_forest_rows,
)
from afg.shared.config import load_corpus_config
from afg.shared.paths import EXPERIMENT_A_DIR, OPENKOS_WORKSPACES_DIR

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 30)
pd.set_option("display.max_colwidth", 90)

config = load_hosted_config()
spec = config.comparison
EVALUATION_SERIES = load_corpus_config()["oe1"]["evaluation_series"]
TAU_KEYS = [f"{t:.2f}" for t in (spec.tau, *spec.sensitivity_taus)]


def interval_args(figure):
    # Keyword arguments of format_interval for an estimate-and-interval figure.
    return {"estimate": figure["estimate"], "interval": figure["interval"]}


display(models_frame(config))

,Etiqueta,Laboratorio,Modelo,Punto de acceso
0,openai-small,OpenAI,gpt-5-mini,https://api.openai.com
1,openai-large,OpenAI,gpt-6-luna,https://api.openai.com
2,anthropic-small,Anthropic,claude-haiku-5-5,https://api.anthropic.com
3,anthropic-large,Anthropic,claude-sonnet-5-5,https://api.anthropic.com
4,google-small,Google,gemini-3.8-flash,https://generativelanguage.googleapis.com/v1beta/openai
5,google-large,Google,gemini-4-argon,https://generativelanguage.googleapis.com/v1beta/openai


In [2]:
n_models = len(config.models)
resamples = format(spec.bootstrap_resamples, ",").replace(",", ".")
alpha = format_es(spec.family_alpha)
margin = format_es(spec.precision_margin)
display(
    Markdown(
        f"**Lectura.** Los seis modelos corren sobre las {len(EVALUATION_SERIES)} series de "
        f"evaluación, con {config.runs_per_model} repeticiones por serie y por modelo, la versión "
        f"{config.required_version} de OpenKOS (instalada aparte de la 0.7.0 de la línea base) y "
        f"`reasoning_effort = {config.reasoning_effort}` en los seis por igual. Los *embeddings* "
        f"siguen siendo `{config.embedding_model}` en el Ollama local. A los modelos de pago no se "
        f"les envían `seed` ni `temperature`: los proveedores los rechazan o los ignoran, y las "
        f"{config.runs_per_model} repeticiones miden esa variabilidad.\n\n"
        f"**Regla de mejora (decisión 7).** Para cada modelo se calcula la diferencia "
        f"(modelo menos línea base) con un bootstrap pareado por reunión ({resamples} "
        f"remuestreos, semilla {spec.bootstrap_seed}) y un nivel Bonferroni por los {n_models} "
        f"modelos (1 menos {alpha}/{n_models}). Un modelo **mejora** la extracción cuando el "
        f"intervalo de la diferencia de recall queda entero sobre 0 y el límite inferior de la "
        f"diferencia de precision no baja de {margin}. Si el intervalo de recall contiene el 0, "
        f"se informa «sin diferencia demostrada»; si queda entero bajo 0, «empeora».\n\n"
        f"**Indicadores de operación (decisión 8).** Tiempo de extracción por reunión (con la "
        f"razón de tiempos frente a la línea base), proporción de extracciones completas, "
        f"decisiones extraídas por reunión y costo en dólares. Son descriptivos."
    )
)

**Lectura.** Los seis modelos corren sobre las 11 series de evaluación, con 3 repeticiones por serie y por modelo, la versión 0.7.2 de OpenKOS (instalada aparte de la 0.7.0 de la línea base) y `reasoning_effort = low` en los seis por igual. Los *embeddings* siguen siendo `bge-m3` en el Ollama local. A los modelos de pago no se les envían `seed` ni `temperature`: los proveedores los rechazan o los ignoran, y las 3 repeticiones miden esa variabilidad.

**Regla de mejora (decisión 7).** Para cada modelo se calcula la diferencia (modelo menos línea base) con un bootstrap pareado por reunión (10.000 remuestreos, semilla 42) y un nivel Bonferroni por los 6 modelos (1 menos 0,05/6). Un modelo **mejora** la extracción cuando el intervalo de la diferencia de recall queda entero sobre 0 y el límite inferior de la diferencia de precision no baja de -0,05. Si el intervalo de recall contiene el 0, se informa «sin diferencia demostrada»; si queda entero bajo 0, «empeora».

**Indicadores de operación (decisión 8).** Tiempo de extracción por reunión (con la razón de tiempos frente a la línea base), proporción de extracciones completas, decisiones extraídas por reunión y costo en dólares. Son descriptivos.

---

# 2. Línea base: `experimento-a-formal`

**Pregunta.** ¿Cuáles son las cifras de la corrida formal con la que se compara y cuánto tarda en
extraer una reunión?

**Qué se hace.** Se leen el `informe.json` publicado de la corrida formal (precision, recall y F1 con
su intervalo de 95 %, agrupando por reunión) y el tiempo de `ingest` de cada reunión, que sale de los
manifiestos de los espacios de trabajo locales.

In [3]:
baseline_name = config.baseline_run_name
informe_path = EXPERIMENT_A_DIR / baseline_name / "informe.json"
informe = json.loads(informe_path.read_text(encoding="utf-8"))
rows = []
for tau in (spec.tau, *spec.sensitivity_taus):
    figures = baseline_figures(informe, tau)
    rows.append(
        {
            "τ": format_es(tau),
            "Precision": format_interval(**interval_args(figures["precision"])),
            "Recall": format_interval(**interval_args(figures["recall"])),
            "F1": format_interval(**interval_args(figures["f1"])),
            "Reuniones": figures["n_meetings"],
        }
    )
display(pd.DataFrame(rows))

,τ,Precision,Recall,F1,Reuniones
0,"0,70","0,15 [0,12; 0,18]","0,31 [0,25; 0,37]","0,20 [0,16; 0,24]",44
1,"0,75","0,10 [0,07; 0,13]","0,21 [0,15; 0,27]","0,13 [0,10; 0,17]",44
2,"0,80","0,05 [0,03; 0,07]","0,10 [0,07; 0,15]","0,07 [0,04; 0,09]",44


In [4]:
baseline_data = load_run_data(
    EXPERIMENT_A_DIR / baseline_name, OPENKOS_WORKSPACES_DIR / baseline_name
)
baseline_time = ingest_time_summary(baseline_data)
if baseline_time["n_meetings"]:
    display(
        Markdown(
            f"**Tiempo de la línea base.** La mediana del `ingest` por reunión es de "
            f"{baseline_time['median_s']:.0f} s, sobre {baseline_time['n_meetings']} reuniones "
            f"(terminaron {baseline_time['complete']} de {baseline_time['expected']} extracciones "
            f"reunión por repetición), en el equipo local de la corrida formal."
        )
    )
else:
    display(
        Markdown(
            "**Pendiente.** No están los manifiestos de los espacios de trabajo de la línea "
            "base (`data/interim/openkos/experimento-a-formal/`, fuera de git); sin ellos no "
            "se puede mostrar el tiempo por reunión ni comparar tiempos en la sección 6."
        )
    )

**Tiempo de la línea base.** La mediana del `ingest` por reunión es de 307 s, sobre 44 reuniones (terminaron 132 de 132 extracciones reunión por repetición), en el equipo local de la corrida formal.

In [5]:
tau_main = baseline_figures(informe, spec.tau)
recall_main, precision_main = tau_main["recall"], tau_main["precision"]
display(
    Markdown(
        f"**Lectura.** Con τ = {format_es(spec.tau)} la línea base encuentra "
        f"{format_es(100 * recall_main['estimate'], 1)} % de las decisiones de la referencia "
        f"(recall {format_interval(**interval_args(recall_main))}) y "
        f"{format_es(100 * precision_main['estimate'], 1)} % de lo que extrae está en la "
        f"referencia (precision). Estas son las cifras que cada modelo de pago debe superar "
        f"según la regla de la sección 1. La corrida formal no se vuelve a correr ni se modifica."
    )
)

**Lectura.** Con τ = 0,70 la línea base encuentra 30,8 % de las decisiones de la referencia (recall 0,31 [0,25; 0,37]) y 14,9 % de lo que extrae está en la referencia (precision). Estas son las cifras que cada modelo de pago debe superar según la regla de la sección 1. La corrida formal no se vuelve a correr ni se modifica.

---

# 3. Puente de continuidad: ¿el cambio de versión altera las cifras?

**Pregunta.** Los modelos de pago corren con OpenKOS 0.7.2 y la línea base con la 0.7.0. ¿Cambia por
sí sola la versión el resultado de `qwen3:8b`?

**Qué se hace.** Se corre `qwen3:8b` con la 0.7.2 en el Ollama local, con la misma configuración de
la línea base, sobre una serie (ES2002) y una repetición. Se acepta que la versión no explica las
diferencias si las tres métricas del puente quedan dentro del rango mínimo–máximo de las tres
repeticiones formales de esa serie (ADR 0026, decisión 6). Si la corrida del puente no existe o aún
no termina, la sección lo dice y no calcula nada.

In [6]:
bridge_info = bridge_state(config, runs_root=EXPERIMENT_A_DIR)
bridge_ready = bridge_info["state"] == "lista"
similarity = make_alignment_similarity()


def reference_of(series_id):
    return load_series_reference(series_id, language="en").decisions


payload = build_comparison(
    config=config,
    labels=list(config.models),
    include_bridge=bridge_ready,
    pilot=False,
    runs_root=EXPERIMENT_A_DIR,
    workspaces_root=OPENKOS_WORKSPACES_DIR,
    reference_of=reference_of,
    similarity=similarity,
)
INSIDE = (
    "El puente queda **dentro** del rango formal en las tres métricas: se acepta que el "
    "cambio de versión de OpenKOS no explica las diferencias entre los modelos y la línea base."
)
OUTSIDE = (
    "El puente queda **fuera** del rango formal en al menos una métrica: el cambio de versión "
    "puede explicar parte de las diferencias, y la comparación de las secciones 5 y 6 pierde "
    "fuerza (ADR 0026, decisión 6)."
)
if bridge_ready:
    bridge = payload["bridge"]
    display(bridge_frame(bridge))
    display(Markdown("**Lectura.** " + (INSIDE if bridge["all_inside"] else OUTSIDE)))
else:
    display(
        Markdown(
            f"**Pendiente** (estado de la corrida del puente: *{bridge_info['state']}*). "
            f"{bridge_info['detail']} Cuando termine, se vuelve a ejecutar este notebook y la "
            f"sección compara `{config.bridge.run_name}` con el rango de las tres repeticiones "
            f"formales de {', '.join(config.bridge.series)}. Mientras tanto, la comparación con "
            f"los modelos de pago debe leerse sin saber si la versión de OpenKOS influye."
        )
    )

**Pendiente** (estado de la corrida del puente: *en curso*). La corrida del puente aún no termina. Cuando termine, se vuelve a ejecutar este notebook y la sección compara `experimento-a-puente-0-7-2` con el rango de las tres repeticiones formales de ES2002. Mientras tanto, la comparación con los modelos de pago debe leerse sin saber si la versión de OpenKOS influye.

---

# 4. Estado de las corridas

**Pregunta.** ¿Qué modelos ya tienen corrida y cuánto cubre cada una?

**Qué se hace.** Para cada modelo se lee el manifiesto de su carpeta. Los estados son: *sin corrida*
(no hay carpeta), *piloto* (solo la corrida de prueba de una serie y una repetición), *parcial*
(corrida completa interrumpida o en curso) y *completa* (todas las series de evaluación con todas las
repeticiones terminadas).

In [7]:
status = pd.DataFrame(
    run_status(config, runs_root=EXPERIMENT_A_DIR, evaluation_series=EVALUATION_SERIES)
).rename(
    columns={
        "label": "Modelo",
        "estado": "Estado",
        "series": "Series cubiertas",
        "corridas": "Repeticiones terminadas por serie",
    }
)
display(status)
counts = status["Estado"].value_counts()
summary = ", ".join(f"{n} modelo(s) {state}" for state, n in counts.items())
display(
    Markdown(
        f"**Lectura.** {summary} (de {len(config.models)}). Las secciones 5 y 6 comparan solo "
        f"las corridas que existen, y para cada modelo solo sobre las series que su corrida cubre."
    )
)

,Modelo,Estado,Series cubiertas,Repeticiones terminadas por serie
0,openai-small,sin corrida,-,-
1,openai-large,sin corrida,-,-
2,anthropic-small,sin corrida,-,-
3,anthropic-large,sin corrida,-,-
4,google-small,sin corrida,-,-
5,google-large,sin corrida,-,-


**Lectura.** 6 modelo(s) sin corrida (de 6). Las secciones 5 y 6 comparan solo las corridas que existen, y para cada modelo solo sobre las series que su corrida cubre.

---

# 5. Calidad de la extracción por modelo

**Pregunta.** ¿Cada modelo de pago extrae más decisiones de la referencia que `qwen3:8b`, sin perder
precision?

**Qué se muestra.** Para cada modelo con corrida: precision, recall y F1 con τ = 0,70 (principal) y,
como sensibilidad, con 0,75 y 0,80; la diferencia frente a la línea base (modelo − línea base) con su
intervalo bootstrap pareado y Bonferroni; y el veredicto de la regla de la sección 1. Una diferencia
positiva en recall significa que el modelo encuentra más decisiones de la referencia. La figura
muestra la diferencia de recall con τ = 0,70: los puntos a la derecha de la línea del 0 son mejoras, y
un intervalo que cruza el 0 no permite afirmar una diferencia.

In [8]:
quality_available = bool(recall_forest_rows(payload))
if not quality_available:
    display(
        Markdown(
            "**Pendiente.** Ningún modelo de pago tiene corrida todavía (sección 4), de modo "
            "que no hay cifras que comparar. Esta sección se llena al volver a ejecutar el "
            "notebook tras las corridas."
        )
    )
for key in TAU_KEYS if quality_available else []:
    role = "principal" if key == TAU_KEYS[0] else "sensibilidad"
    display(Markdown(f"**τ = {key.replace('.', ',')}** ({role})"))
    display(quality_table(payload, key))

**Pendiente.** Ningún modelo de pago tiene corrida todavía (sección 4), de modo que no hay cifras que comparar. Esta sección se llena al volver a ejecutar el notebook tras las corridas.

In [9]:
COLORS = {
    "mejora": "#009E73",
    "empeora": "#D55E00",
    "sin diferencia demostrada": "#7F7F7F",
}
forest = recall_forest_rows(payload)
if forest:
    fig, ax = plt.subplots(figsize=(7, 0.55 * len(forest) + 1.6))
    for i, row in enumerate(forest):
        ax.errorbar(
            row["estimate"],
            i,
            xerr=[[row["estimate"] - row["low"]], [row["high"] - row["estimate"]]],
            fmt="o",
            color=COLORS.get(row["verdict"], "#7F7F7F"),
            capsize=3,
        )
    ax.axvline(0, color="black", linewidth=1)
    ax.set_yticks(range(len(forest)), [row["label"] for row in forest])
    ax.invert_yaxis()
    ax.set_xlabel("Diferencia de recall (modelo menos línea base)")
    ax.set_title(f"Recall frente a la línea base, τ = {format_es(spec.tau)}")
    handles = [
        plt.Line2D([], [], marker="o", linestyle="", color=c, label=v) for v, c in COLORS.items()
    ]
    ax.legend(handles=handles, loc="best", fontsize=8)
    fig.tight_layout()
    plt.show()
else:
    display(Markdown("**Pendiente.** La figura se dibuja cuando haya al menos una corrida."))

**Pendiente.** La figura se dibuja cuando haya al menos una corrida.

---

# 6. Operación: tiempo, extracciones completas, decisiones y costo

**Pregunta.** Además de extraer mejor o peor, ¿los modelos de pago son más rápidos o más confiables?

**Qué se muestra.** Para cada modelo con corrida: la mediana del tiempo de `ingest` por reunión y la
de la línea base; la razón de tiempos (modelo ÷ línea base) con su intervalo bootstrap pareado, calculado
sobre el logaritmo de la razón, con el veredicto *más rápido* (intervalo entero bajo 1), *más lento*
(entero sobre 1) o sin diferencia demostrada; la proporción de reuniones cuyo `ingest` terminó con
extracción; las decisiones extraídas por reunión (ayudan a leer la precision); y el costo en dólares
cuando se registró en la configuración. **Lectura del tiempo:** la línea base corre en un equipo local
y los modelos de pago en la infraestructura del proveedor; la razón mide la experiencia de uso de cada
opción, no la eficiencia del modelo.

In [10]:
if quality_available:
    display(operation_table(payload))
else:
    display(
        Markdown(
            "**Pendiente.** Sin corridas de modelos de pago no hay tiempos, extracciones "
            "completas, decisiones por reunión ni costos que mostrar."
        )
    )

**Pendiente.** Sin corridas de modelos de pago no hay tiempos, extracciones completas, decisiones por reunión ni costos que mostrar.

In [11]:
TIME_COLORS = {"más rápido": "#009E73", "más lento": "#D55E00"}
time_rows = time_forest_rows(payload)
if time_rows:
    fig, ax = plt.subplots(figsize=(7, 0.55 * len(time_rows) + 1.6))
    for i, row in enumerate(time_rows):
        ax.errorbar(
            row["estimate"],
            i,
            xerr=[[row["estimate"] - row["low"]], [row["high"] - row["estimate"]]],
            fmt="o",
            color=TIME_COLORS.get(row["verdict"], "#7F7F7F"),
            capsize=3,
        )
    ax.axvline(1, color="black", linewidth=1)
    ax.set_xscale("log")
    ax.set_yticks(range(len(time_rows)), [row["label"] for row in time_rows])
    ax.invert_yaxis()
    ax.set_xlabel("Razón de tiempos de extracción (modelo / línea base, escala logarítmica)")
    ax.set_title("Tiempo por reunión frente a la línea base (1 = igual)")
    fig.tight_layout()
    plt.show()
else:
    display(Markdown("**Pendiente.** La figura se dibuja cuando haya una corrida con tiempos."))

**Pendiente.** La figura se dibuja cuando haya una corrida con tiempos.

---

# 7. Lectura y límites

**Cómo leer los resultados.** Estas son las reglas de lectura que valen para todas las cifras de las
secciones 5 y 6:

- **Es una extensión descriptiva.** Las series de evaluación ya se usaron en la corrida formal y el
  equipo ya vio sus cifras. Ningún resultado de este notebook confirma una hipótesis de la tesis; el
  veredicto de cada modelo (*mejora*, *sin diferencia demostrada*, *empeora*) describe lo observado.
- **Qué se mide.** Solo la extracción de decisiones de C2. No se mide si C2 responde mejor ni se usa el
  juez ni el Experimento B: la pregunta es si un modelo más capaz extrae más decisiones.
- **Los modelos de pago cambian sin aviso detrás del mismo identificador.** Una cifra vale para el
  modelo tal como se sirvió en la fecha de su corrida. Si un identificador dejó de existir, se reemplazó
  por el sucesor del mismo nivel (ADR 0026, decisión 3) y debe quedar escrito en el manifiesto de la
  corrida y en esta sección.
- **`reasoning_effort = low` en los seis modelos.** El ajuste es el mismo para no tratar distinto a un
  laboratorio, pero no es el óptimo de ninguno; el resultado vale con ese ajuste.
- **Sin `seed` ni `temperature` en los modelos de pago.** Las tres repeticiones miden la variabilidad
  que eso introduce; los intervalos ya la incluyen porque cada reunión aporta el promedio de sus
  repeticiones.
- **El tiempo depende del equipo.** La línea base corre en un equipo local y los modelos de pago en la
  infraestructura del proveedor, con carga y redes distintas. La razón de tiempos describe la
  experiencia de uso, no la velocidad intrínseca del modelo.
- **Cobertura.** Cada comparación se restringe a las series que cubre la corrida del modelo (sección 4);
  un modelo con corrida parcial se compara sobre menos reuniones y su intervalo es más ancho.
- **El puente de continuidad (sección 3)** decide cuánto peso tiene la comparación: si queda fuera del
  rango formal, el cambio de versión de OpenKOS es una explicación posible de las diferencias.
- **Datos a terceros.** Las transcripciones de AMI (CC BY 4.0, sin datos personales sensibles) se envían
  a OpenAI, Anthropic y Google; es la primera vez que el proyecto lo hace (ADR 0026, consecuencias).

**Conclusiones.** Esta sección se completa tras la corrida completa (T8). *Marcador: hasta entonces no se
escribe ninguna conclusión, porque todavía no hay corridas completas que leer.*